In [1]:
import socket
import threading
import os
import struct
import zlib
import json
import io
import tkinter as tk
from tkinter.scrolledtext import ScrolledText
from tkinter import filedialog
from PIL import Image, ImageTk

In [2]:
HOST = '0.0.0.0'
PORT = 8080

SAVE_DIR = "server_received_files"
os.makedirs(SAVE_DIR, exist_ok=True)

MSG_TEXT = 'TEXT'
MSG_FILE = 'FILE'
MSG_INFO = 'INFO'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}
VIDEO_EXTS = {'.mp4', '.avi', '.mkv', '.mov', '.webm', '.flv'}

clients   = []
usernames = {}
lock      = threading.Lock()
gui       = None   # set after GUI is created

In [3]:
def send_packet(conn, msg_type, payload_bytes, meta=None):
    header       = {'type': msg_type, 'length': len(payload_bytes), 'meta': meta or {}}
    header_bytes = json.dumps(header).encode('utf-8')
    conn.sendall(struct.pack('>I', len(header_bytes)))
    conn.sendall(header_bytes)
    conn.sendall(payload_bytes)


def recv_exact(conn, n):
    data = b''
    while len(data) < n:
        chunk = conn.recv(n - len(data))
        if not chunk:
            raise ConnectionError("Connection closed")
        data += chunk
    return data


def recv_packet(conn):
    header_len = struct.unpack('>I', recv_exact(conn, 4))[0]
    header     = json.loads(recv_exact(conn, header_len).decode('utf-8'))
    payload    = recv_exact(conn, header['length'])
    return header['type'], payload, header.get('meta', {})

In [4]:
def broadcast(msg_type, payload_bytes, meta=None, exclude_conn=None):
    with lock:
        targets = [c for c in clients if c != exclude_conn]
    for client in targets:
        try:
            send_packet(client, msg_type, payload_bytes, meta)
        except:
            remove_client(client)


def remove_client(conn):
    with lock:
        if conn not in clients:
            return
        username = usernames.pop(conn, 'Unknown')
        clients.remove(conn)
    try:
        conn.close()
    except:
        pass
    broadcast(MSG_INFO, f"[-] {username} left the room.".encode('utf-8'))
    print(f"[DISCONNECTED] {username}")

In [5]:
def compress_data(data: bytes) -> bytes:
    return zlib.compress(data, level=6)


def decompress_data(data: bytes) -> bytes:
    return zlib.decompress(data)


def should_compress(ext: str) -> bool:
    # Videos are already compressed — skip
    if ext in VIDEO_EXTS:
        return False
    # Images and small files benefit from compression
    return True

In [6]:
def display_image_in_widget(chat_box, image_data: bytes, label: str, root):
    """Show a thumbnail inside a ScrolledText. Must be called from the main thread."""
    try:
        img = Image.open(io.BytesIO(image_data))
        img.thumbnail((220, 220))
        photo = ImageTk.PhotoImage(img)

        chat_box.config(state=tk.NORMAL)
        chat_box.insert(tk.END, f"{label}\n")
        chat_box.image_create(tk.END, image=photo)
        chat_box.insert(tk.END, "\n")
        chat_box.config(state=tk.DISABLED)
        chat_box.yview(tk.END)

        # Keep reference so image isn't garbage-collected
        if not hasattr(chat_box, '_images'):
            chat_box._images = []
        chat_box._images.append(photo)
    except Exception as e:
        chat_box.config(state=tk.NORMAL)
        chat_box.insert(tk.END, f"{label} [File saved]\n")
        chat_box.config(state=tk.DISABLED)
        chat_box.yview(tk.END)

In [7]:
def handle_client(conn, addr):
    print(f"[NEW] {addr} connected")
    username = f"Guest_{addr[1]}"
    try:
        # First packet = username
        msg_type, payload, meta = recv_packet(conn)
        username = payload.decode('utf-8').strip() or username

        with lock:
            usernames[conn] = username
            clients.append(conn)

        print(f"[JOIN] {username}")
        broadcast(MSG_INFO, f"[+] {username} joined!".encode('utf-8'), exclude_conn=conn)

        if gui:
            gui.log(f"[+] {username} joined", tag='info')
            gui.update_users()

        while True:
            msg_type, payload, meta = recv_packet(conn)

            # ── Text ──────────────────────────────────────────────
            if msg_type == MSG_TEXT:
                text = payload.decode('utf-8')
                full = f"{username}: {text}"
                print(f"[MSG] {full}")
                broadcast(MSG_TEXT, full.encode('utf-8'), exclude_conn=conn)
                if gui:
                    gui.log(full, tag='other')

            # ── File (image / video / any) ─────────────────────────
            elif msg_type == MSG_FILE:
                filename   = meta.get('filename', 'unknown')
                ext        = meta.get('ext', '')
                compressed = meta.get('compressed', False)
                file_type  = meta.get('file_type', 'file')
                orig_size  = meta.get('original_size', 0)

                # Decompress to get the real file data for saving/displaying
                file_data  = decompress_data(payload) if compressed else payload
                comp_ratio = round((1 - len(payload) / orig_size) * 100, 1) if compressed and orig_size else 0

                # Save on server
                save_path = os.path.join(SAVE_DIR, f"{username}_{filename}")
                with open(save_path, 'wb') as f:
                    f.write(file_data)
                print(f"[FILE] {username} -> '{filename}' | {len(file_data)//1024} KB | compression: {comp_ratio}%")

                # ✅ Forward the ORIGINAL payload (still compressed) to other clients
                fwd_meta           = dict(meta)
                fwd_meta['sender'] = username
                broadcast(MSG_FILE, payload, meta=fwd_meta, exclude_conn=conn)

                # Show in server GUI
                if gui:
                    if file_type == 'image':
                        # Schedule image display on main thread
                        gui.root.after(0, lambda d=file_data, u=username, fn=filename:
                            display_image_in_widget(
                                gui.chat_box, d,
                                f"{u} sent image '{fn}':",
                                gui.root
                            )
                        )
                    else:
                        size_mb = round(len(file_data) / (1024*1024), 2)
                        gui.log(
                            f"{'🎬' if file_type == 'video' else ''} "
                            f"{username} sent '{filename}' | {size_mb} MB — saved to {save_path}",
                            tag='system'
                        )

    except Exception as e:
        print(f"[ERROR] {addr}: {e}")
    finally:
        remove_client(conn)
        if gui:
            gui.log(f"[-] {username} left", tag='system')
            gui.update_users()

In [8]:
class ServerGUI:

    def __init__(self):
        self.root = tk.Tk()
        self._build_gui()

    def _build_gui(self):
        self.root.title("Chat Server — Admin Panel")
        self.root.configure(bg='#1e1e2e')
        self.root.geometry('580x640')
        self.root.resizable(True, True)

        # Header
        tk.Label(
            self.root, text="🖥️  Server Admin Panel",
            bg='#313244', fg='#f38ba8',
            font=('Segoe UI', 13, 'bold'), pady=8
        ).pack(fill=tk.X)

        # Users bar
        self.users_label = tk.Label(
            self.root, text="Connected users: 0",
            bg='#1e1e2e', fg='#a6e3a1',
            font=('Segoe UI', 9), anchor='w', padx=10
        )
        self.users_label.pack(fill=tk.X, pady=(4, 0))

        # Chat log
        self.chat_box = ScrolledText(
            self.root, state=tk.DISABLED,
            bg='#181825', fg='#cdd6f4',
            font=('Segoe UI', 10), relief=tk.FLAT,
            wrap=tk.WORD, padx=10, pady=10
        )
        self.chat_box.pack(fill=tk.BOTH, expand=True, padx=8, pady=(4, 4))
        self.chat_box.tag_config('info',   foreground='#a6e3a1')
        self.chat_box.tag_config('admin',  foreground='#f38ba8')
        self.chat_box.tag_config('other',  foreground='#cdd6f4')
        self.chat_box.tag_config('system', foreground='#fab387')

        # Message input row
        row1 = tk.Frame(self.root, bg='#1e1e2e')
        row1.pack(fill=tk.X, padx=8, pady=(0, 4))

        self.entry = tk.Entry(
            row1, bg='#313244', fg='#cdd6f4',
            insertbackground='white', relief=tk.FLAT, font=('Segoe UI', 10)
        )
        self.entry.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=6, padx=(0, 6))
        self.entry.bind('<Return>', lambda e: self.send_text())

        btn = dict(relief=tk.FLAT, font=('Segoe UI', 9, 'bold'), padx=10, pady=6)
        tk.Button(row1, text='Send',    command=self.send_text,  bg='#f38ba8', fg='#1e1e2e', **btn).pack(side=tk.LEFT, padx=(0, 4))
        tk.Button(row1, text='File', command=self.send_file,  bg='#a6e3a1', fg='#1e1e2e', **btn).pack(side=tk.LEFT, padx=(0, 4))
        tk.Button(row1, text='Users',command=self.show_users, bg='#89b4fa', fg='#1e1e2e', **btn).pack(side=tk.LEFT)

        # Kick row
        row2 = tk.Frame(self.root, bg='#1e1e2e')
        row2.pack(fill=tk.X, padx=8, pady=(0, 8))

        self.kick_entry = tk.Entry(
            row2, bg='#313244', fg='#6c7086',
            insertbackground='white', relief=tk.FLAT, font=('Segoe UI', 10)
        )
        self.kick_entry.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=5, padx=(0, 6))
        self.kick_entry.insert(0, 'Username to kick...')
        self.kick_entry.bind('<FocusIn>',  lambda e: self._clear_ph())
        self.kick_entry.bind('<FocusOut>', lambda e: self._restore_ph())
        tk.Button(row2, text='Kick', command=self.kick_user, bg='#f38ba8', fg='#1e1e2e', **btn).pack(side=tk.LEFT)

    def _clear_ph(self):
        if self.kick_entry.get() == 'Username to kick...':
            self.kick_entry.config(fg='#cdd6f4')
            self.kick_entry.delete(0, tk.END)

    def _restore_ph(self):
        if not self.kick_entry.get():
            self.kick_entry.config(fg='#6c7086')
            self.kick_entry.insert(0, 'Username to kick...')

    def log(self, text: str, tag: str = 'other'):
        self.root.after(0, lambda: self._append(text, tag))

    def _append(self, text: str, tag: str):
        self.chat_box.config(state=tk.NORMAL)
        self.chat_box.insert(tk.END, text + '\n', tag)
        self.chat_box.config(state=tk.DISABLED)
        self.chat_box.yview(tk.END)

    def update_users(self):
        with lock:
            names = list(usernames.values())
        self.root.after(0, lambda: self.users_label.config(
            text=f"Connected: {len(names)}  |  {', '.join(names) if names else '—'}"
        ))

    def send_text(self):
        msg = self.entry.get().strip()
        if not msg:
            return
        broadcast(MSG_TEXT, f"[ADMIN]: {msg}".encode('utf-8'))
        self._append(f"You (Admin): {msg}", tag='admin')
        self.entry.delete(0, tk.END)

    def send_file(self):
        filepath = filedialog.askopenfilename(
            title='Select file to broadcast',
            filetypes=[
                ('All Files', '*.*'),
                ('Images',    '*.jpg *.jpeg *.png *.gif *.bmp *.webp'),
                ('Videos',    '*.mp4 *.avi *.mkv *.mov *.webm *.flv'),
            ]
        )
        if not filepath:
            return
        try:
            filename  = os.path.basename(filepath)
            ext       = os.path.splitext(filename)[1].lower()
            file_type = 'image' if ext in IMAGE_EXTS else 'video' if ext in VIDEO_EXTS else 'file'

            with open(filepath, 'rb') as f:
                raw_data = f.read()

            orig_size   = len(raw_data)
            do_compress = should_compress(ext)

            payload = compress_data(raw_data) if do_compress else raw_data

            if do_compress:
                comp_ratio = round((1 - len(payload) / orig_size) * 100, 1)
                self._append(
                    f"Sending '{filename}' | {orig_size//1024} KB → "
                    f"{len(payload)//1024} KB (Compressed {comp_ratio}%)", tag='system'
                )
            else:
                self._append(
                    f"Sending '{filename}' | {round(orig_size/(1024*1024),2)} MB", tag='system'
                )

            meta = {
                'filename'      : filename,
                'file_type'     : file_type,
                'compressed'    : do_compress,
                'original_size' : orig_size,
                'ext'           : ext,
                'sender'        : 'ADMIN'
            }

            broadcast(MSG_FILE, payload, meta=meta)

            if file_type == 'image':
                self.root.after(0, lambda d=raw_data, fn=filename:
                    display_image_in_widget(
                        self.chat_box, d,
                        f"You (Admin) sent image '{fn}':",
                        self.root
                    )
                )
            elif file_type == 'video':
                self._append(f"You sent video: '{filename}' to all clients.", tag='admin')
            else:
                self._append(f"You sent file: '{filename}' to all clients.", tag='admin')

        except Exception as e:
            self._append(f"[File Error: {e}]", tag='system')

    # ── Show users ─────────────────────────────────────────────
    def show_users(self):
        with lock:
            names = list(usernames.values())
        if names:
            self._append(f"Online ({len(names)}): {', '.join(names)}", tag='info')
        else:
            self._append("No clients connected.", tag='system')

    def kick_user(self):
        target = self.kick_entry.get().strip()
        if not target or target == 'Username to kick...':
            return
        with lock:
            conn = next((c for c, n in usernames.items() if n == target), None)
        if conn:
            remove_client(conn)
            self._append(f"'{target}' was kicked.", tag='system')
            self.update_users()
        else:
            self._append(f"[!] User '{target}' not found.", tag='system')
        self.kick_entry.delete(0, tk.END)

In [ ]:
import subprocess, sys
try:
    from PIL import Image
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'Pillow', '-q'])
    from PIL import Image

def accept_loop(server):
    while True:
        try:
            conn, addr = server.accept()
            t = threading.Thread(target=handle_client, args=(conn, addr), daemon=True)
            t.start()
        except:
            break

def start_server_gui():
    global gui

    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, PORT))
    server.listen()
    print(f"[START] Server on {HOST}:{PORT}")

    # Accept clients in background
    threading.Thread(target=accept_loop, args=(server,), daemon=True).start()

    # Build GUI on main thread
    gui = ServerGUI()
    gui.log(f"Server running on {HOST}:{PORT}", tag='info')
    gui.log(f"Files saved to: {os.path.abspath(SAVE_DIR)}", tag='info')
    gui.root.mainloop()

    server.close()
    print("[STOP] Server stopped.")

start_server_gui()

[START] Server on 0.0.0.0:8080
